# Day 13 — Broadcasting & Vectorised Operations
### Python for Data Science · Module 1 · Topic 1.12

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

**Session length:** 2 hours
**Format:** 90 min concepts + live coding · 30 min practice

| # | What we cover | Time |
|---|---|---|
| 1 | Vectorisation — element-wise operations and ufuncs | 20 min |
| 2 | Broadcasting — the two rules | 30 min |
| 3 | Aggregation and the axis rule | 20 min |
| 4 | Vectorised logic — `where`, `clip`, `any`, `all` | 15 min |
| 5 | Mini build: scaling a dataset two ways | 5 min |
| 6 | **Practice notebook (separate file)** | 30 min |

> **Why this session pays off later.** Broadcasting is not a NumPy curiosity — it is how
> pandas subtracts a column mean from a DataFrame, how scikit-learn scales features, and how
> every neural network adds a bias vector to a batch of inputs. The shapes get bigger; the
> rule stays exactly the same.

---
## 0. Yesterday's open question

Day 12's final challenge did this:

```python
data    = np.array([[10., 200., 3000.],
                    [20., 400., 1000.],
                    [30., 600., 2000.]])
col_min = data.min(axis=0)      # shape (3,)
scaled  = (data - col_min) / (col_max - col_min)
```

`data` is `(3, 3)`. `col_min` is `(3,)`. **Different shapes — and yet no error.**

The answer has a name: **broadcasting**. Let's start there.

In [ ]:
import numpy as np
print("numpy", np.__version__)

data = np.array([[10., 200., 3000.],
                 [20., 400., 1000.],
                 [30., 600., 2000.]])
col_min = data.min(axis=0)

print("data.shape    :", data.shape)
print("col_min.shape :", col_min.shape)
print("subtracting anyway:")
print(data - col_min)

---
# 1. Vectorisation

## 1.1 Every operator works element by element

In [ ]:
a = np.array([1, 2, 3])
b = np.array([10, 20, 30])

print("a + b  :", a + b)
print("a - b  :", a - b)
print("a * b  :", a * b)
print("a / b  :", a / b)
print("a ** 2 :", a ** 2)
print("a > 2  :", a > 2)
print("a == b :", a == b)

### ⚠️ `a * b` is NOT matrix multiplication

In [ ]:
print("a * b (element-wise):", a * b)     # [10 40 90]
print("a @ b (dot product) :", a @ b)     # 140  = 1*10 + 2*20 + 3*30

# You will need @ in Module 4. Until then, * is what you want.

## 1.2 Universal functions (ufuncs)

In [ ]:
x = np.array([1., 4., 9.])

print("np.sqrt :", np.sqrt(x))
print("np.exp  :", np.exp(np.array([0., 1.])).round(4))
print("np.log  :", np.log(np.array([1., np.e])).round(4))
print("np.abs  :", np.abs(np.array([-1, 2, -3])))
print("np.round:", np.round(np.array([1.234, 5.678]), 2))

In [ ]:
# math.sqrt works on ONE number and fails on an array
import math
try:
    math.sqrt(x)
except TypeError as e:
    print("math.sqrt(array) -> TypeError:", str(e)[:55])
    print("  -> once you import numpy, reach for np.sqrt")

---
# 2. Broadcasting

## 2.1 The two rules

> **Line the shapes up from the RIGHT. Two dimensions are compatible when they are equal,
> or when one of them is 1.**
>
> A shorter shape is padded on the **left** with 1s. Wherever a dimension is 1, that array
> is stretched to match — without ever being copied in memory.

```
  (3, 4)        (3, 4)        (3, 4)        (3, 1)        (3, 4)
     (4,)          (3,)       (3, 1)          (1, 4)        scalar
  --------      --------      --------      --------      --------
  (3, 4)  OK    ERROR         (3, 4)  OK    (3, 4)  OK    (3, 4)  OK
                4 vs 3
```

In [4]:
import numpy as np

def try_broadcast(shape_a, shape_b):
    a, b = np.ones(shape_a), np.ones(shape_b)
    try:
        print(a)
        print(b)
        print(f"  {str(shape_a):10} + {str(shape_b):10} ->  {(a + b).shape}")
        print(a+b)
    except ValueError as e:
        print(f"  {str(shape_a):10} + {str(shape_b):10} ->  ERROR: {str(e)[-24:]}")

print("compatible:")
try_broadcast((3, 4), (4,))
try_broadcast((3, 4), (3, 1))
try_broadcast((3, 1), (1, 4))
try_broadcast((2, 3, 4), (3, 4))

print("\nincompatible:")
try_broadcast((3, 4), (3,))
try_broadcast((4,), (3,))

compatible:
[[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]]
[1. 1. 1. 1.]
  (3, 4)     + (4,)       ->  (3, 4)
[[2. 2. 2. 2.]
 [2. 2. 2. 2.]
 [2. 2. 2. 2.]]
[[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]]
[[1.]
 [1.]
 [1.]]
  (3, 4)     + (3, 1)     ->  (3, 4)
[[2. 2. 2. 2.]
 [2. 2. 2. 2.]
 [2. 2. 2. 2.]]
[[1.]
 [1.]
 [1.]]
[[1. 1. 1. 1.]]
  (3, 1)     + (1, 4)     ->  (3, 4)
[[2. 2. 2. 2.]
 [2. 2. 2. 2.]
 [2. 2. 2. 2.]]
[[[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]

 [[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]]
[[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]]
  (2, 3, 4)  + (3, 4)     ->  (2, 3, 4)
[[[2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]]

 [[2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]]]

incompatible:
[[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]]
[1. 1. 1.]
  (3, 4)     + (3,)       ->  ERROR:  with shapes (3,4) (3,) 
[1. 1. 1. 1.]
[1. 1. 1.]
  (4,)       + (3,)       ->  ERROR: r with shapes (4,) (3,) 


## 2.2 What "stretching" actually looks like

```
   data (3,3)          col_min (3,)         result (3,3)
  [  10  200 3000 ]                        [   0    0  2000 ]
  [  20  400 1000 ]  -  [ 10 200 1000 ]  = [  10  200     0 ]
  [  30  600 2000 ]      stretched down    [  20  400  1000 ]
```

**The stretched rows are not real.** NumPy never builds them — it just reads the same three
numbers again for each row.

> **This is why broadcasting is free.** Subtracting a column mean from a million-row table
> does not create a million copies of the mean. The memory cost is zero.

In [ ]:
# Scalars are the simplest broadcast of all
arr = np.array([1, 2, 3])
print("arr * 2   :", arr * 2)      # the 2 is stretched to [2 2 2]
print("arr + 100 :", arr + 100)

# Every operation from Day 12 was broadcasting in disguise
marks = np.array([88, 35, 71])
print("marks >= 40:", marks >= 40)  # 40 stretched across all three

## 2.3 When it fails — and how to read the error

In [ ]:
a = np.ones((3, 4))
b = np.ones(3)

try:
    a + b
except ValueError as e:
    print("ValueError:", e)
    print()
    print("  It tells you BOTH shapes. Align them from the right:")
    print("     (3, 4)")
    print("        (3,)")
    print("     4 against 3 - neither is 1, so they clash.")

### The fix: say which axis you meant

In [ ]:
b = np.array([1., 2., 3.])
print("b.shape                :", b.shape)
print("b.reshape(-1, 1).shape :", b.reshape(-1, 1).shape)
print("b[:, np.newaxis].shape :", b[:, np.newaxis].shape)
print("b[:, None].shape       :", b[:, None].shape)      # shorthand

a = np.ones((3, 4))
print()
print("a + b.reshape(-1, 1) works, shape:", (a + b.reshape(-1, 1)).shape)

**A `(3,)` is a row by default.** Making it `(3, 1)` declares it a column.

> ### The three-step debug, every time
> 1. Print both `.shape` values
> 2. Align them from the right
> 3. Ask which one should be stretched
>
> Nearly every shape error in NumPy, pandas and scikit-learn is solved by those three steps.

In [ ]:
# The same (3,) behaves completely differently depending on its shape
v = np.arange(3)
print("as a row    (3,)  :", (v + v).shape, "->", v + v)
print("as a column (3,1) :", (v[:, None] + v).shape, "  <- a 3x3 grid!")
print()
print(v[:, None] + v)

---
# 3. Aggregation — the axis you name disappears

In [ ]:
m = np.array([[1, 2, 3],
              [4, 5, 6]])

print("m.sum()        :", m.sum())          # everything -> one number
print("m.sum(axis=0)  :", m.sum(axis=0))    # down the rows  -> per column
print("m.sum(axis=1)  :", m.sum(axis=1))    # across columns -> per row

In [ ]:
# The rule that removes the confusion
print("m.shape             :", m.shape)          # (2, 3)
print("m.sum(axis=0).shape :", m.sum(axis=0).shape)   # (3,) - the 2 went
print("m.sum(axis=1).shape :", m.sum(axis=1).shape)   # (2,) - the 3 went

# Forget "rows" and "columns". Look at what DISAPPEARS from the shape.

In [ ]:
m = np.array([[1, 2, 3], [4, 5, 6]])

print("mean   :", m.mean(axis=0))
print("min    :", m.min(axis=1))
print("max    :", m.max())
print("argmax :", m.argmax(), " <- the POSITION in the flattened array, not the value")
print("argmax per row:", m.argmax(axis=1))
print("cumsum :", m.cumsum())          # running total

### ⚠️ `std` defaults to the **population** formula

In [ ]:
arr = np.array([1., 2., 3., 4.])

print("arr.std()        :", arr.std().round(4))         # divides by n
print("arr.std(ddof=1)  :", arr.std(ddof=1).round(4))   # divides by n-1

# NumPy defaults to ddof=0 (population); pandas defaults to ddof=1 (sample).
# Module 2 will care about this difference - it is worth remembering now.

---
# 4. Vectorised logic

In [ ]:
marks = np.array([88, 35, 71, 22, 64])

# np.where - a vectorised if/else
print("labels:", np.where(marks >= 40, "pass", "fail"))

# Day 6's  "A if cond else B"  applied to every element.
# Like that one, NOTHING is dropped - the result is the same length as the input.
print("bumped:", np.where(marks < 40, 40, marks))   # raise every fail to 40

In [ ]:
# np.clip - squeeze everything into a range
print("original:", marks)
print("clipped :", np.clip(marks, 40, 80))

#  88 came down to 80
#  35 and 22 came up to 40
#  71 and 64 were already inside

In [ ]:
# any and all - collapse a mask into ONE answer
print("(marks > 90).any() :", (marks > 90).any())
print("(marks > 20).all() :", (marks > 20).all())
print("(marks < 40).sum() :", (marks < 40).sum())

# These are what you use inside an if statement:
if (marks < 40).any():
    print("\nAt least one student failed.")

# A raw mask cannot go in an if - that was yesterday's
# "truth value of an array is ambiguous" error.

---
# 5. Putting it together — scaling a dataset two ways

In [ ]:
# 3 samples, 3 features - on wildly different scales
data = np.array([[10., 200., 3000.],
                 [20., 400., 1000.],
                 [30., 600., 2000.]])

# ---- 1. Min-max scaling: squeeze every column into 0-1
col_min = data.min(axis=0)          # (3,) one per column
col_max = data.max(axis=0)
minmax  = (data - col_min) / (col_max - col_min)

# ---- 2. Standardisation: mean 0, standard deviation 1
col_mean = data.mean(axis=0)
col_std  = data.std(axis=0)
z = (data - col_mean) / col_std

print("min-max:")
print(minmax)
print("\nz-score:")
print(z.round(3))

In [ ]:
# Proof it worked - without a single loop
print("minmax range :", minmax.min(), "to", minmax.max())
print("z means      :", z.mean(axis=0).round(10))
print("z stds       :", z.std(axis=0).round(10))

- **Broadcasting, twice** — `(3,3)` minus `(3,)` in both formulas
- **`axis=0`** — one value per column, as always
- **No loops at all** — nine numbers, one expression
- **Min-max** — fixed 0–1 range
- **Z-score** — mean 0, std 1, handles outliers better
- **Verified, not assumed** — the proof cell above

> These two lines are the whole of feature scaling. In Module 3 scikit-learn does it for you
> as `MinMaxScaler` and `StandardScaler` — and now you know exactly what they are doing.

---
# 6. Recap — the twelve things to remember

1. Every operator works element by element, with no loop.
2. `a * b` multiplies positions; `a @ b` is the dot product.
3. Use `np.sqrt`, not `math.sqrt` — `math` wants one number.
4. **Broadcasting:** align the shapes from the **right**.
5. Two dimensions fit if equal, or if one of them is 1.
6. A shorter shape is padded on the **left** with 1s.
7. Nothing is copied — the small array is simply re-read.
8. A `(3,)` is a row. Use `reshape(-1, 1)` to make a column.
9. On any shape error: print both shapes, align right.
10. The axis you name is the one that disappears.
11. `argmax` gives a **position**; `std` defaults to `ddof=0`.
12. `np.where` is if/else; `any()` and `all()` feed an `if`.

---

### 📝 Now open **`Day13_Practice_Questions.ipynb`** for the 30-minute practice session.

### Homework
- Standardise a 10×4 random array and prove each column has mean 0.
- Build a 10×10 multiplication table with broadcasting and no loops.
- Write down three shape pairs that will **not** broadcast, and why.

### Next class — Topic 1.13: NumPy statistical functions
`mean`, `median`, `percentile`, correlation — and handling missing values with the
nan-aware functions.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*